# Hyperparameter Tuning (W&B)

Systematically search over learning rate, regularization, and model capacity using **Weights & Biases Bayesian sweeps** on the TinyShakespeare character-level GPT.

## Prerequisites

- Run `2. Preprocessing.ipynb` first — outputs in `data/artifacts/`.
- Optional: review `3. Baseline Model Architecture.ipynb` for the same GPT design.
- Install W&B: `pip install wandb`, then `wandb login`.

## Setup

Import dependencies, resolve the project root, and define `BASE_CONFIG`.

**Fixes applied:**
- `Optional[Tensor]` instead of `Tensor | None` (Python 3.9 compat)
- `lr_scheduler.LRScheduler` instead of deprecated `_LRScheduler`
- `torch.load(..., weights_only=True)` to suppress PyTorch 2.x warning

In [ ]:
from pathlib import Path
from typing import Optional
import random
import time
import math
import json

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import wandb
except ImportError as e:
    raise ImportError("Install wandb: pip install wandb") from e

cwd = Path.cwd()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(f"Could not locate a 'data' directory from cwd: {cwd}")

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = DATA_DIR / "artifacts"

BASE_CONFIG = {
    "seed": 42,
    "block_size": 128,
    "batch_size": 32,
    "max_steps": 2000,
    "eval_interval": 200,
    "learning_rate": 3e-4,
    "weight_decay": 0.1,
    "betas": (0.9, 0.95),
    "warmup_steps": 100,
    "min_lr": 1e-5,
    "n_layers": 4,
    "d_model": 128,
    "n_heads": 4,
    "d_ff": 4,
    "dropout": 0.2,
    "wandb_project": "genre-story-generator",
    "wandb_entity": None,
    "wandb_group": "sweep_lr_dropout",
    "wandb_job_type": "training-sweep",
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
set_seed(BASE_CONFIG["seed"])


## Load Artifacts

Load preprocessed token IDs from notebook 2.

**Fix:** `torch.load(..., weights_only=True)` — suppresses deprecation warning in PyTorch >= 2.0.

In [ ]:
train_ids_path = ARTIFACTS_DIR / "train_ids.pt"
val_ids_path   = ARTIFACTS_DIR / "val_ids.pt"
vocab_path     = ARTIFACTS_DIR / "char_vocab.json"

assert train_ids_path.exists(), f"Missing {train_ids_path}. Run preprocessing notebook first."
assert val_ids_path.exists(),   f"Missing {val_ids_path}. Run preprocessing notebook first."
assert vocab_path.exists(),     f"Missing {vocab_path}. Run preprocessing notebook first."

# weights_only=True — required in PyTorch >= 2.0 to avoid FutureWarning / security warning
train_ids = torch.load(train_ids_path, weights_only=True)
val_ids   = torch.load(val_ids_path,   weights_only=True)

with open(vocab_path, "r", encoding="utf-8") as f:
    vocab_payload = json.load(f)

vocab_size = vocab_payload["vocab_size"]
print(f"train_ids: {train_ids.shape} | val_ids: {val_ids.shape} | vocab: {vocab_size}")


def get_batch(split: str, batch_size: int, block_size: int, device: str = device):
    data = train_ids if split == "train" else val_ids if split == "val" else None
    if data is None:
        raise ValueError("split must be 'train' or 'val'")
    if len(data) <= block_size:
        raise ValueError(f"{split} split too short for block_size={block_size}")

    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i     : i + block_size    ] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


## Model Architecture

Compact **decoder-only GPT**: `tokens → embeddings → N × Transformer blocks → LayerNorm → linear head`

**Fixes:**
- `_causal_mask`: removed broken `register_buffer` / reassign pattern; now creates a fresh causal mask per call if size changes (cached as plain attribute, not W&B buffer)
- `Optional[torch.Tensor]` type hint instead of `Tensor | None` (Python 3.9 compat)

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads

        self.qkv_proj    = nn.Linear(d_model, 3 * d_model)
        self.out_proj    = nn.Linear(d_model, d_model)
        self.attn_dropout  = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # FIX: use plain Python attribute instead of register_buffer(None) + later reassign.
        # register_buffer with None then self.mask = tensor bypasses the module buffer registry
        # and causes device-placement bugs. Cache as a plain attribute instead.
        self._causal_cache: Optional[torch.Tensor] = None

    def _causal_mask(self, size: int, device: torch.device) -> torch.Tensor:
        if self._causal_cache is None or self._causal_cache.size(-1) < size:
            m = torch.tril(torch.ones(size, size, device=device)).view(1, 1, size, size)
            self._causal_cache = m
        return self._causal_cache[:, :, :size, :size]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()
        q, k, v = self.qkv_proj(x).chunk(3, dim=-1)

        def reshape(t: torch.Tensor) -> torch.Tensor:
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q, k, v = reshape(q), reshape(k), reshape(v)

        att  = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = self._causal_mask(T, x.device)
        att  = att.masked_fill(mask == 0, float("-inf"))
        att  = self.attn_dropout(F.softmax(att, dim=-1))

        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(y))


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff_mult: int, dropout: float):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_ff_mult * d_model),
            nn.GELU(),
            nn.Linear(d_ff_mult * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size: int, config: dict):
        super().__init__()
        d_model    = config["d_model"]
        n_heads    = config["n_heads"]
        n_layers   = config["n_layers"]
        dropout    = config["dropout"]
        block_size = config["block_size"]
        d_ff_mult  = config.get("d_ff", 4)

        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding   = nn.Embedding(block_size, d_model)
        self.drop   = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff_mult, dropout) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        idx: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ):
        B, T = idx.size()
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} exceeds block_size {self.block_size}")

        pos    = torch.arange(T, device=idx.device).unsqueeze(0)
        x      = self.drop(self.token_embedding(idx) + self.pos_embedding(pos))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


## Learning Rate Schedule

**Linear warmup → cosine decay** down to `min_lr`.

**Fix:** Inherit from `torch.optim.lr_scheduler.LRScheduler` (public API, PyTorch >= 2.0). Fall back to `_LRScheduler` for PyTorch < 2.0.

In [ ]:
# FIX: _LRScheduler is deprecated in PyTorch >= 2.0; use the public LRScheduler alias.
# This try/except keeps backward compat with PyTorch 1.x.
try:
    _LRSchedulerBase = torch.optim.lr_scheduler.LRScheduler
except AttributeError:
    _LRSchedulerBase = torch.optim.lr_scheduler._LRScheduler  # type: ignore[attr-defined]


class CosineWarmupScheduler(_LRSchedulerBase):
    def __init__(
        self,
        optimizer: torch.optim.Optimizer,
        warmup_steps: int,
        max_steps: int,
        min_lr: float = 0.0,
        last_epoch: int = -1,
    ):
        self.warmup_steps = warmup_steps
        self.max_steps    = max_steps
        self.min_lr       = min_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        step = self.last_epoch + 1
        if step < self.warmup_steps:
            scale = step / max(1, self.warmup_steps)
        else:
            progress = (step - self.warmup_steps) / max(1, self.max_steps - self.warmup_steps)
            progress = min(max(progress, 0.0), 1.0)
            scale    = 0.5 * (1.0 + math.cos(math.pi * progress))
        return [self.min_lr + (base_lr - self.min_lr) * scale for base_lr in self.base_lrs]


## Training Loop

| Function | Role |
|----------|------|
| `create_model_and_optimizer` | Build GPT + AdamW (decoupled weight decay) |
| `evaluate_loss` | Mean validation cross-entropy |
| `train` | W&B sweep callback — logs metrics, saves best checkpoint |

**Fix:** `wandb.config` is a `wandb.Config` object, not a dict — wrap with `dict()` before merging to avoid key-access errors.

In [ ]:
def create_model_and_optimizer(config: dict):
    model = GPTLanguageModel(vocab_size=vocab_size, config=config).to(device)

    decay_params, no_decay_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.dim() >= 2 and "bias" not in name and "norm" not in name.lower():
            decay_params.append(p)
        else:
            no_decay_params.append(p)

    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params,    "weight_decay": config["weight_decay"]},
            {"params": no_decay_params, "weight_decay": 0.0},
        ],
        lr=config["learning_rate"],
        betas=tuple(config.get("betas", (0.9, 0.95))),
    )

    scheduler = CosineWarmupScheduler(
        optimizer,
        warmup_steps=config["warmup_steps"],
        max_steps=config["max_steps"],
        min_lr=config.get("min_lr", 0.0),
    )
    return model, optimizer, scheduler


@torch.no_grad()
def evaluate_loss(model: nn.Module, config: dict, num_batches: int = 50) -> float:
    model.eval()
    losses = [
        model(
            *get_batch("val", config["batch_size"], config["block_size"])
        )[1].item()
        for _ in range(num_batches)
    ]
    model.train()
    return float(np.mean(losses))


def train(config: Optional[dict] = None) -> None:
    run = wandb.init(
        project  = BASE_CONFIG["wandb_project"],
        entity   = BASE_CONFIG.get("wandb_entity"),
        group    = BASE_CONFIG.get("wandb_group"),
        job_type = BASE_CONFIG.get("wandb_job_type", "training-sweep"),
        config   = BASE_CONFIG,
    )

    # FIX: wandb.config is a wandb.Config object; dict() converts it safely.
    # Merge order: BASE_CONFIG < passed-in config < sweep-sampled wandb.config.
    merged = {**BASE_CONFIG, **(config or {}), **dict(wandb.config)}
    set_seed(merged["seed"])

    model, optimizer, scheduler = create_model_and_optimizer(merged)
    wandb.log({"model/parameters": sum(p.numel() for p in model.parameters())})

    best_val_loss = float("inf")
    best_state    = None
    start_time    = time.time()

    for step in range(merged["max_steps"]):
        x, y = get_batch("train", merged["batch_size"], merged["block_size"])
        _, loss = model(x, y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        tokens_seen = (step + 1) * merged["batch_size"] * merged["block_size"]
        elapsed     = time.time() - start_time

        log_payload = {
            "train/loss"           : loss.item(),
            "train/step"           : step,
            "train/tokens"         : tokens_seen,
            "train/tokens_per_sec" : tokens_seen / max(elapsed, 1e-6),
            "optim/lr"             : optimizer.param_groups[0]["lr"],
        }

        if (step + 1) % merged["eval_interval"] == 0 or step == merged["max_steps"] - 1:
            val_loss = evaluate_loss(model, merged)
            log_payload["val/loss"]        = val_loss
            log_payload["val/perplexity"]  = math.exp(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state    = {"model": model.state_dict(), "config": merged}

        wandb.log(log_payload)

    if best_state is not None:
        ckpt_dir  = DATA_DIR / "checkpoints"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        ckpt_path = ckpt_dir / f"gpt_sweep_{run.id}.pt"
        torch.save(best_state, ckpt_path)

        artifact = wandb.Artifact(f"tinyshakespeare-char-gpt-{run.id}", type="model")
        artifact.add_file(str(ckpt_path))
        artifact.add_file(str(vocab_path))
        run.log_artifact(artifact)

    run.finish()


## W&B Sweep Configuration

Bayesian optimization (`method: bayes`) minimizes **`val/loss`**.

| Parameter | Search | Notes |
|-----------|--------|-------|
| `learning_rate` | log-uniform [1e-4, 1e-2] | AdamW step size |
| `n_layers` | {4, 5, 6, 7, 8} | Model depth |
| `d_model` | {128, 256, 384, 512} | Hidden size |
| `n_heads` | {4, 8} | Attention heads |
| `dropout` | uniform [0.1, 0.3] | Regularization |
| `weight_decay` | {0.05, 0.1, 0.2} | AdamW decay |
| `warmup_steps` | {50, 100, 200} | LR warmup length |

In [ ]:
sweep_config = {
    "method": "bayes",
    "metric": {"name": "val/loss", "goal": "minimize"},
    "parameters": {
        "learning_rate": {"distribution": "log_uniform_values", "min": 1e-4,  "max": 1e-2},
        "n_layers"     : {"values": [4, 5, 6, 7, 8]},
        "d_model"      : {"values": [128, 256, 384, 512]},
        "n_heads"      : {"values": [4, 8]},
        "dropout"      : {"distribution": "uniform", "min": 0.1, "max": 0.3},
        "weight_decay" : {"values": [0.05, 0.1, 0.2]},
        "warmup_steps" : {"values": [50, 100, 200]},
    },
}


## Create the Sweep

Registers the search space on W&B and returns a `sweep_id`. Re-run only when you change `sweep_config`.

In [ ]:
sweep_id = wandb.sweep(
    sweep_config,
    project=BASE_CONFIG["wandb_project"],
    entity=BASE_CONFIG.get("wandb_entity"),
)
print("sweep_id:", sweep_id)


## Run the Sweep Agent

`count=10` runs 10 trials. Monitor on the W&B dashboard URL printed above.

In [ ]:
wandb.agent(sweep_id, function=train, count=10)


## Analyze Sweep Results

Fetch all runs, build a leaderboard, print best hyperparameters, plot sensitivity charts.

**Fixes:**
- `steps[eval_mask]` crash when `steps = range(...)` — now always a `pd.Series`
- `plt.style` fallback for older matplotlib
- `_step` column checked before use

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# FIX: seaborn-v0_8-darkgrid only exists in matplotlib >= 3.6; fallback gracefully
try:
    plt.style.use("seaborn-v0_8-darkgrid")
except OSError:
    try:
        plt.style.use("seaborn-darkgrid")
    except OSError:
        pass  # use default style

SWEEP_METRIC = "val/loss"
SWEEP_PARAMS = [
    "learning_rate", "n_layers", "d_model",
    "n_heads", "dropout", "weight_decay", "warmup_steps",
]


def _sweep_api_path(sweep_id: str) -> str:
    project = BASE_CONFIG["wandb_project"]
    entity  = BASE_CONFIG.get("wandb_entity") or wandb.Api().default_entity
    return f"{entity}/{project}/{sweep_id}"


def fetch_sweep_runs(sweep_id: str) -> pd.DataFrame:
    api   = wandb.Api()
    sweep = api.sweep(_sweep_api_path(sweep_id))
    rows  = []

    for run in sweep.runs:
        hist = run.history(keys=[SWEEP_METRIC, "val/perplexity"])

        if not hist.empty and SWEEP_METRIC in hist.columns and hist[SWEEP_METRIC].notna().any():
            best_idx = hist[SWEEP_METRIC].idxmin()
            best_val = float(hist.loc[best_idx, SWEEP_METRIC])
            best_ppl = (
                float(hist.loc[best_idx, "val/perplexity"])
                if "val/perplexity" in hist.columns and pd.notna(hist.loc[best_idx, "val/perplexity"])
                else math.exp(best_val)
            )
        elif run.summary.get(SWEEP_METRIC) is not None:
            best_val = float(run.summary[SWEEP_METRIC])
            best_ppl = float(run.summary.get("val/perplexity") or math.exp(best_val))
        else:
            continue

        row = {
            "run_id"            : run.id,
            "run_name"          : run.name,
            "state"             : run.state,
            SWEEP_METRIC        : best_val,
            "val/perplexity"    : best_ppl,
            "model/parameters"  : run.summary.get("model/parameters"),
        }
        for param in SWEEP_PARAMS:
            row[param] = run.config.get(param)
        rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    sort_cols  = [SWEEP_METRIC]
    ascending  = [True]
    if df["model/parameters"].notna().any():
        sort_cols.append("model/parameters")
        ascending.append(True)

    return df.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)


runs_df = fetch_sweep_runs(sweep_id)
print(f"Fetched {len(runs_df)} runs from sweep '{sweep_id}'\n")

if runs_df.empty:
    print("No runs with val/loss found. Run the sweep agent cell first.")
else:
    table_cols = ["run_name", "state", SWEEP_METRIC, "val/perplexity", "model/parameters"] + SWEEP_PARAMS
    display(runs_df[table_cols])

    best = runs_df.iloc[0]
    params_fmt = int(best["model/parameters"]) if pd.notna(best.get("model/parameters")) else "n/a"
    print("\n--- Best run (lowest val/loss; tie-break: fewer parameters) ---")
    print(f"  name  : {best['run_name']}  ({best['run_id']})")
    print(f"  loss  : {best[SWEEP_METRIC]:.4f}  |  perplexity = {best['val/perplexity']:.2f}  |  params = {params_fmt}")
    print("\n  hyperparameters:")
    for param in SWEEP_PARAMS:
        val = best[param]
        fmt = f"{val:.6g}" if param == "learning_rate" and val is not None else str(val)
        print(f"    {param}: {fmt}")


In [ ]:
if runs_df.empty:
    print("Nothing to plot — fetch runs first.")
else:
    best = runs_df.iloc[0]

    # ── Sensitivity: categorical hyperparameters ───────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(f"Sweep '{sweep_id}': mean {SWEEP_METRIC} by hyperparameter", fontsize=13)

    for ax, param in zip(axes, ["d_model", "n_layers", "n_heads"]):
        summary = runs_df.groupby(param, observed=True)[SWEEP_METRIC].agg(["mean", "count"])
        summary["mean"].plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
        ax.set_title(param); ax.set_xlabel(param); ax.set_ylabel(f"mean {SWEEP_METRIC}")
        ax.tick_params(axis="x", rotation=0)
        for i, (_, row) in enumerate(summary.iterrows()):
            ax.text(i, row["mean"], f"n={int(row['count'])}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout(); plt.show()

    # ── Sensitivity: continuous hyperparameters ────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].scatter(runs_df["learning_rate"], runs_df[SWEEP_METRIC], alpha=0.8, s=60)
    axes[0].set_xscale("log"); axes[0].set_xlabel("learning_rate")
    axes[0].set_ylabel(SWEEP_METRIC); axes[0].set_title("Learning rate vs val/loss")

    axes[1].scatter(runs_df["dropout"], runs_df[SWEEP_METRIC], alpha=0.8, s=60, color="darkorange")
    axes[1].set_xlabel("dropout"); axes[1].set_ylabel(SWEEP_METRIC)
    axes[1].set_title("Dropout vs val/loss")
    plt.tight_layout(); plt.show()

    # ── Training curve for best run ────────────────────────────────────────────
    api      = wandb.Api()
    proj_path = "/".join(_sweep_api_path(sweep_id).split("/")[:2])  # entity/project
    best_run  = api.run(f"{proj_path}/{best['run_id']}")
    hist      = best_run.history(keys=["train/loss", SWEEP_METRIC, "optim/lr"])

    # FIX: hist._step may not exist; use reset_index to get a safe integer index
    # FIX: steps must be a pd.Series (not range) so boolean indexing works
    steps    = hist["_step"].reset_index(drop=True) if "_step" in hist.columns else pd.Series(range(len(hist)))
    tl       = hist["train/loss"].reset_index(drop=True)
    vl       = hist[SWEEP_METRIC].reset_index(drop=True)
    eval_mask = vl.notna()

    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(steps, tl, label="train/loss", alpha=0.85)
    ax1.set_xlabel("step"); ax1.set_ylabel("train/loss", color="C0")
    ax1.tick_params(axis="y", labelcolor="C0")

    ax2 = ax1.twinx()
    # FIX: steps[eval_mask] now works because steps is pd.Series, not range()
    ax2.plot(steps[eval_mask], vl[eval_mask], "o-", color="C1", label="val/loss", linewidth=2)
    ax2.set_ylabel("val/loss", color="C1")
    ax2.tick_params(axis="y", labelcolor="C1")

    fig.suptitle(f"Best run: {best['run_name']}  ({best['run_id']})")
    fig.tight_layout(); plt.show()


## Notes

- Requires `data/artifacts/` from notebook 2.
- Best checkpoint per trial saved to `data/checkpoints/gpt_sweep_<run_id>.pt`
- W&B artifact: `tinyshakespeare-char-gpt-<run_id>`
- Full leaderboard also available on the [W&B sweep page](https://wandb.ai).